# How to select products contaminated with RFI

### 1. Packages and Libraries

In [10]:
%pip install aiohttp requests
%load_ext autoreload
%autoreload 2
    
from shapely.geometry import Polygon, shape, mapping
from rasterio.windows import Window
import xml.etree.ElementTree as ET
import matplotlib.pyplot as plt
from pystac_client import Client
from folium import GeoJson
import geopandas as gpd
from io import BytesIO
import rasterio as rio
from tqdm import tqdm
from PIL import Image
import pandas as pd
import numpy as np
import requests
import pathlib
import fsspec
import folium
import fiona
import os
import json
from datetime import datetime, timedelta
import time
import antimeridian
import yaml
from pathlib import Path

Note: you may need to restart the kernel to use updated packages.
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### 2. Query the catalog

Query a large amount of products to identify those affected by RFI. The typical filters of STAC can be used (by location, time, ids, etc.). Working with L1A data is prefered.

#### 2.1 Search in the catalog

In [5]:
catalog_url = 'https://catalog.maap.eo.esa.int/catalogue/'
catalog = Client.open(catalog_url)

search = catalog.search(
    collections = ["BiomassLevel1a"],
    datetime = ['2026-01-01T00:00:00Z','2026-07-01T23:59:59Z'],
    max_items = 100,
    limit = 200,
)

items = list(search.items())
print(f"Accessing {len(items)} items (limited by max_items).")
print(f"{search.matched()} items found that matched the query.")

Accessing 100 items (limited by max_items).
232177 items found that matched the query.


#### 2.2 Inspect the searched items

Create the GeoDataframe to plot the footprints of the selected items

In [6]:
features = []
for it in items:
    if it.geometry is None:
        raise ValueError(f"Item {it.id} has no geometry (footprint).")
    props = it.properties or {}
    features.append({
        "type": "Feature",
        "geometry": it.geometry,
        "properties": {
            "product_id": it.id,
            "start_datetime": props.get("start_datetime"),
            "end_datetime": props.get("end_datetime"),
        },
    })


gdf = gpd.GeoDataFrame.from_features(features, crs="EPSG:4326")
gdf = gdf.reset_index()
gdf = gdf[["index", "product_id", "start_datetime", "end_datetime", "geometry"]]
gdf["geometry"] = gdf.geometry.apply(lambda g: shape(antimeridian.fix_shape(mapping(g))))
gdf.head(len(items))

,index,product_id,start_datetime,end_datetime,geometry
0,0,BIO_S2_SCS__1S_20260701T123849_20260701T123909...,2026-07-01T12:38:49.864Z,2026-07-01T12:39:09.638Z,"POLYGON ((80.88125 -14.48715, 81.47012 -14.602..."
1,1,BIO_S2_SCS__1S_20260701T123830_20260701T123851...,2026-07-01T12:38:30.847Z,2026-07-01T12:38:51.249Z,"POLYGON ((81.1171 -13.37077, 81.70216 -13.4858..."
2,2,BIO_S2_SCS__1S_20260701T123811_20260701T123832...,2026-07-01T12:38:11.831Z,2026-07-01T12:38:32.232Z,"POLYGON ((81.36048 -12.21592, 81.94188 -12.331..."
3,3,BIO_S2_SCS__1S_20260701T123752_20260701T123813...,2026-07-01T12:37:52.816Z,2026-07-01T12:38:13.217Z,"POLYGON ((81.60335 -11.06099, 82.18138 -11.176..."
4,4,BIO_S2_SCS__1S_20260701T123733_20260701T123754...,2026-07-01T12:37:33.803Z,2026-07-01T12:37:54.202Z,"POLYGON ((81.84579 -9.90591, 82.42077 -10.0217..."
...,...,...,...,...,...
95,95,BIO_S2_SCS__1S_20260701T092708_20260701T092729...,2026-07-01T09:27:08.817Z,2026-07-01T09:27:29.232Z,"POLYGON ((126.13885 -31.8087, 126.81774 -31.92..."
96,96,BIO_S2_SCS__1S_20260701T092649_20260701T092710...,2026-07-01T09:26:49.788Z,2026-07-01T09:27:10.203Z,"POLYGON ((126.4147 -30.6597, 127.08365 -30.772..."
97,97,BIO_S2_SCS__1S_20260701T092630_20260701T092651...,2026-07-01T09:26:30.759Z,2026-07-01T09:26:51.173Z,"POLYGON ((126.68533 -29.5098, 127.3456 -29.622..."
98,98,BIO_S2_SCS__1S_20260701T092611_20260701T092632...,2026-07-01T09:26:11.732Z,2026-07-01T09:26:32.145Z,"POLYGON ((126.95694 -28.36019, 127.60649 -28.4..."


#### 2.3 Plot the footprints

In [7]:
## Ensure both are WGS84 for folium
gdf = gdf.to_crs("EPSG:4326")

# Center/zoom using bounds
bounds = gpd.GeoSeries(list(gdf.geometry), crs="EPSG:4326")
minx, miny, maxx, maxy = bounds.total_bounds
center = [(miny + maxy) / 2, (minx + maxx) / 2]

m = folium.Map(location=center, tiles="OpenStreetMap", zoom_start=3)

def style(_):
    return {"color": "#ff7f0e", "weight": 1, "fillOpacity": 0.10}  

GeoJson(
    gdf.__geo_interface__,
    name=f"Products ({len(gdf)})",
    tooltip=folium.GeoJsonTooltip(fields=["product_id", "index"]),
    style_function=style,
).add_to(m)


folium.LayerControl(collapsed=False).add_to(m)
m.fit_bounds([[miny, minx], [maxy, maxx]])
m

### 3 Select products with RFI

Stream the annotation file of the selected items and search for the rfiDetectionFlag. If true, the product is affected by RFI. If so, check the rfiCorrectionFlag which is up when mitigation is applied, otherwise the RFI is intact. This is driven by the geospatial map (RFI mitigation applied on desert areas and Asia).

Save the RFI-related attributes:
- Percentage of bandwidth affected
- Percentage of lines affected
- Max bandwidth

Only the stats related to the Frequency Domain Notch Filter are selected because if RFI is detected on with the Time Domain algorithm it will also affect the time domain given the nature of RFIs in this band.

Additionally, due to the nature of the detection algorithm implemented in the BPS, it is possible to have a large False Alarm rate, meaning many products will be identified as RFI-affected but they might not. For that reason, the user is able to select a min percentage of affected bandwidth and/or a min percentage of affected lines to avoid false alarm which normally occupy the minimum bandwidth.

In [8]:
# This is permanently valid, you don't need to change the two lines below.
CLIENT_ID="offline-token"
CLIENT_SECRET="p1eL7uonXs6MDxtGbgKdPVRAmnGxHpVE"

Insert the path to your token file from the config.ymal

In [13]:
# Load the config.yaml to access the paths
cfg_file = Path("config.yaml").resolve()
cfg = yaml.safe_load(cfg_file.read_text())

# Set the name of the path in the config.yaml
token_path = (cfg_file.parent / cfg["paths"]["token_file"]).resolve()
out_path = (cfg_file.parent / cfg["paths"]["output"]).resolve()

print(f"Token Path: {token_path}")
print(f"Output Path: {out_path}")

Token Path: /home/jovyan/Biomass_hackathon/token.txt
Output Path: /home/jovyan/biomass-hackathon-2026/topics/02-rfi-removal/out


In [15]:
if pathlib.Path(token_path).exists():
  with open(token_path, "rt") as f:
    OFFLINE_TOKEN = f.read().strip().replace("\n","")
    url = "https://iam.maap.eo.esa.int/realms/esa-maap/protocol/openid-connect/token"
    data = {
        "client_id": CLIENT_ID,
        "client_secret": CLIENT_SECRET,
        "grant_type": "refresh_token",
        "refresh_token": OFFLINE_TOKEN,
        "scope": "offline_access openid"
    }

    response = requests.post(url, data=data)
    response.raise_for_status()

    response_json = response.json()
    access_token = response_json.get('access_token')
    print("Access token retrieved successfully.")

    if not access_token:
        raise RuntimeError("Failed to retrieve access token from IAM response")

else:
    raise RuntimeError(f"No token file found in {os.getcwd()} !")

Access token retrieved successfully.


In [20]:
from src.explore_annotation import read_annot_file

In [21]:
# Parallelize the stream of the annotation files
import threading, time, json
from concurrent.futures import ThreadPoolExecutor
from aiohttp import ClientResponseError

tok = {"v": access_token}
lock = threading.Lock()

def refresh_access_token():
    r = requests.post(
        "https://iam.maap.eo.esa.int/realms/esa-maap/protocol/openid-connect/token",
        data={"client_id": CLIENT_ID, "client_secret": CLIENT_SECRET,
              "grant_type": "refresh_token", "refresh_token": OFFLINE_TOKEN,
              "scope": "offline_access openid"})
    r.raise_for_status()
    return r.json()["access_token"]

def annot_job(it, retries=3):
    """Returns (id, results, error). Never raises, so one bad product can't kill the batch."""
    for attempt in range(retries):
        used = tok["v"]
        try:
            if not it.assets or "enclosure_annot_xml" not in it.assets:
                raise ValueError(f"Item {it.id} has no annotation asset.")
            res = read_annot_file(it.id, it.assets["enclosure_annot_xml"].href, used)
            return it.id, res, None
        except ClientResponseError as e:
            if e.status in (401, 403):
                with lock:
                    if tok["v"] == used:            # only the first thread refreshes
                        tok["v"] = refresh_access_token()
            time.sleep(2 ** attempt)                # back off 1, 2, 4 s
            err = repr(e)
        except Exception as e:
            time.sleep(2 ** attempt)
            err = repr(e)
    return it.id, [], err

N_workers = 8 # Adapt to the number of physical cores of your machine

with ThreadPoolExecutor(max_workers=N_workers) as ex:
    results = list(tqdm(ex.map(annot_job, items), total=len(items)))

products_rfi = [r for _, res, err in results for r in res]
failed = {pid: err for pid, _, err in results if err}
print(len(products_rfi), "RFI products,", len(failed), "failed")

  1%|          | 1/100 [00:00<00:37,  2.63it/s]

rfiDetectionFlag = true
rfiCorrectionFlag = true
rfiDetectionFlag = true
rfiCorrectionFlag = true
rfiDetectionFlag = true
rfiCorrectionFlag = true
rfiDetectionFlag = true
rfiCorrectionFlag = true
rfiDetectionFlag = true
rfiCorrectionFlag = true


  2%|▏         | 2/100 [00:00<00:46,  2.12it/s]

rfiDetectionFlag = true
rfiCorrectionFlag = true
rfiDetectionFlag = true
rfiCorrectionFlag = true
rfiDetectionFlag = true
rfiCorrectionFlag = true
rfiDetectionFlag = true
rfiCorrectionFlag = true


  4%|▍         | 4/100 [00:01<00:33,  2.89it/s]

...

rfiDetectionFlag = true
rfiCorrectionFlag = true
rfiDetectionFlag = true
rfiCorrectionFlag = true


 93%|█████████▎| 93/100 [00:12<00:00,  8.53it/s]

rfiDetectionFlag = true
rfiCorrectionFlag = true
rfiDetectionFlag = true
rfiCorrectionFlag = true
rfiDetectionFlag = true
rfiCorrectionFlag = false
rfiDetectionFlag = true
rfiCorrectionFlag = false
rfiDetectionFlag = true
rfiCorrectionFlag = false
rfiDetectionFlag = true
rfiCorrectionFlag = false
rfiDetectionFlag = true
rfiCorrectionFlag = false


100%|██████████| 100/100 [00:14<00:00,  7.08it/s]

rfiDetectionFlag = true
rfiCorrectionFlag = false
100 RFI products, 0 failed


In [22]:
# save, right after the products_rfi loop
with open(f"{out_path}/products_rfi.json", "w") as f:
    json.dump(products_rfi, f, indent=2)